# Extracción de features (forma, color, textura)

Lee `preprocessed2/manifest.csv`, y para cada imagen carga `_preprocessed.png` (sin blur) y `_mask.png`.
Guarda **un CSV por split**: `features_train.csv` y `features_val.csv` en `OUT_DIR`.

Todas las funciones están en `features.py` (misma carpeta que este notebook). Los parámetros de las features
(`KMEANS_K`, `RIDGE_THR`, `EXTENDED_SHAPE`, Gabor...) se cambian allí; aquí solo van las rutas y la ejecución.

- La **primera columna es `name`** (nombre de la imagen). Después: `class`, `subtype`, `label` (nevus=0, others=1)
  y dos flags de calidad `qc_*` (no son features: no las uses para entrenar).
- Las features llevan prefijo `shape_`, `color_`, `tex_`, `gcolor_` o `gtex_` para poder hacer ablación por familias:
  `shape_` (52), `color_` (82) y `tex_` (104, incluye `tex_lndp_*` y `tex_gabor_*`) se calculan **solo dentro de la lesión (máscara)**;
  `gcolor_` (70) y `gtex_` (104, incluye `gtex_lndp_*` y `gtex_gabor_*`) se calculan sobre **toda la imagen**.
- Total esperado: **412 features**.
- Si la máscara de una imagen está vacía o falla una familia, esas columnas quedan en NaN (las de imagen entera se calculan igualmente).

In [ ]:
from pathlib import Path

import pandas as pd
from joblib import Parallel, delayed
from tqdm.auto import tqdm

import features as F

# ---------------- Configuración ----------------
ROOT = Path(r"C:\Users\Lenovo\PycharmProjects\Melanoma-Challenge-ML-UdG")
PROC_DIR = ROOT / "preprocessed2"
OUT_DIR = ROOT / "features_final"    # carpeta nueva para no pisar los CSV anteriores (420 features)
N_JOBS = 4                           # núcleos (-1 = todos, -2 = todos menos uno)
LIMIT = None                         # p. ej. 20 para probar con pocas imágenes; None = todas

In [ ]:
manifest = pd.read_csv(PROC_DIR / "manifest.csv")
manifest = manifest[manifest["status"].isin(["ok", "skipped"])].reset_index(drop=True)

if LIMIT:
    manifest = manifest.groupby("split").head(LIMIT).reset_index(drop=True)
print("processed:", PROC_DIR)
print(manifest.groupby(["split", "class"]).size().to_string())

## Extracción (paralela)

In [ ]:
gen = Parallel(n_jobs=N_JOBS, return_as="generator")(delayed(F.extract_one)(r) for r in manifest.to_dict("records"))
df = pd.DataFrame([r for r in tqdm(gen, total=len(manifest))])

meta = ["name", "class", "subtype", "label", "qc_seg_ok", "qc_crop_fallback"]
feat_cols = [c for p in F.FEATURE_PREFIXES for c in df.columns if c.startswith(p)]
df = df[meta + feat_cols + ["qc_errors"]]
df["_split"] = df["name"].map(manifest.set_index("name")["split"])
print(f"\n{df.shape[0]} imágenes, {len(feat_cols)} features")

## Guardado: un CSV por split

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

for split in ["train", "val"]:
    part = df[df["_split"] == split].drop(columns="_split")
    if len(part) == 0:
        continue
    path = OUT_DIR / f"features_{split}.csv"
    part.to_csv(path, index=False)
    print(f"{path}  ->  {part.shape[0]} filas x {part.shape[1]} columnas")

# resumen por familia (esperado: shape 52, color 82, tex 104, gcolor 70, gtex 104)
nan_pct = df[feat_cols].isna().mean()
for p in ["shape", "color", "tex", "gcolor", "gtex"]:
    cols_p = [c for c in feat_cols if c.startswith(p + "_")]
    if cols_p:
        print(f"{p:6s} {len(cols_p):3d} columnas | NaN medio: {100 * nan_pct[cols_p].mean():.2f}%")
    else:
        print(f"{p:6s}   0 columnas  <-- ¡la familia falló en todas las imágenes! mira qc_errors")
bad = df[df["qc_errors"] != ""]
print(f"Imágenes con algún error: {len(bad)}")
if len(bad):
    print(bad["qc_errors"].str.split(";").explode().str.replace(r"^(\w+:\w+@\w+:L\d+).*", r"\1", regex=True)
          .value_counts().head(10).to_string())          # errores más frecuentes (tipo, función y línea)
    print(bad[["name", "qc_errors"]].head(5).to_string(index=False))